# 05 - Retrieval: Finding the Right Context

Retrieval is the **bottleneck** of a RAG system. If you retrieve the wrong chunks, even the best LLM will produce a bad answer. Garbage in, garbage out.

This notebook compares three retrieval strategies:
1. **Similarity search** — straightforward top-k nearest neighbors
2. **MMR (Maximal Marginal Relevance)** — balances relevance with diversity
3. **Similarity with score threshold** — filters by minimum similarity

In [ ]:
import os, sys, tempfile
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))

from dotenv import load_dotenv
load_dotenv(os.path.join(os.path.dirname(os.getcwd()), ".env"))

from rag_engine.loaders import load_documents
from rag_engine.chunking.strategies import chunk_documents
from rag_engine.vectorstore.chroma_store import add_documents
from rag_engine.retrieval.retriever import RetrieverFactory

DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), "data", "sample")

## Setup: Build the Vector Store

In [ ]:
# Load, chunk, and embed
md_docs = load_documents(os.path.join(DATA_DIR, "rag_survey.md"))
csv_docs = load_documents(os.path.join(DATA_DIR, "ml_glossary.csv"))
all_docs = md_docs + csv_docs

chunks = chunk_documents(all_docs, strategy="recursive", chunk_size=256, chunk_overlap=30)
print(f"Total chunks: {len(chunks)}")

temp_dir = tempfile.mkdtemp()
vectorstore = add_documents(chunks, collection_name="nb05", persist_directory=temp_dir)

## Strategy 1: Similarity Search (Top-K)

The simplest approach: embed the query, find the k nearest vectors.

In [ ]:
query = "What are the different retrieval strategies in RAG?"

sim_retriever = RetrieverFactory.create_retriever(vectorstore, strategy="similarity", top_k=5)
sim_results = sim_retriever.invoke(query)

print(f"=== Similarity Search (top-5) ===")
for i, doc in enumerate(sim_results, 1):
    print(f"\n[{i}] {doc.page_content[:150]}...")
    print(f"    Source: {doc.metadata.get('source', 'N/A')}")

## Strategy 2: MMR (Maximal Marginal Relevance)

MMR balances **relevance** (how similar to the query) with **diversity** (how different from already-selected documents). This avoids returning 5 near-identical chunks.

The `lambda_mult` parameter controls the trade-off:
- `lambda_mult=1.0` → pure relevance (same as similarity search)
- `lambda_mult=0.0` → pure diversity
- `lambda_mult=0.5` → balanced (recommended)

In [ ]:
mmr_retriever = RetrieverFactory.create_retriever(
    vectorstore, strategy="mmr", top_k=5, lambda_mult=0.5
)
mmr_results = mmr_retriever.invoke(query)

print(f"=== MMR Search (top-5, lambda=0.5) ===")
for i, doc in enumerate(mmr_results, 1):
    print(f"\n[{i}] {doc.page_content[:150]}...")
    print(f"    Source: {doc.metadata.get('source', 'N/A')}")

## Side-by-Side Comparison

Let's compare what each strategy retrieves for the same query.

In [ ]:
import pandas as pd

comparison = []
for i in range(min(len(sim_results), len(mmr_results))):
    comparison.append({
        "Rank": i + 1,
        "Similarity": sim_results[i].page_content[:80] + "...",
        "MMR": mmr_results[i].page_content[:80] + "...",
    })

df = pd.DataFrame(comparison)
print(df.to_string(index=False))

## When to Use Which Strategy

| Strategy | Best for | Trade-off |
|----------|---------|----------|
| **Similarity** | Simple Q&A, when you want the most relevant chunks | May return near-duplicates |
| **MMR** | Complex questions spanning multiple topics | Sacrifices some relevance for diversity |
| **Score threshold** | When you want to avoid low-quality results | May return fewer than k results |

**Key insight:** For most RAG applications, **MMR with lambda_mult=0.5** is a good default. It ensures the LLM gets diverse context covering different aspects of the question.

**Next:** [06 - Building the RAG Chain](./06_rag_chain.ipynb) — wiring retrieval into a complete pipeline.